# Lesson 3A: Neural Networks Theory

## Introduction

### What is a Neuron?

Before we dive into neural networks, let's understand what we mean by a "neuron" in machine learning.

A neuron is inspired by biological neurons in our brains. Just as a biological neuron:
- Receives multiple input signals from other neurons
- Combines these signals
- Decides whether to "fire" (activate) based on the combined signal
- Sends its output to other neurons

An artificial neuron does something remarkably similar:
1. **Receives inputs**: Takes in multiple feature values (x₁, x₂, ..., xₙ)
2. **Weights the inputs**: Multiplies each input by a learned weight (w₁, w₂, ..., wₙ)
3. **Combines them**: Sums all weighted inputs plus a bias term: z = w₁x₁ + w₂x₂ + ... + wₙxₙ + b
4. **Applies activation**: Transforms the sum through a function (like sigmoid: σ(z) = 1/(1+e⁻ᶻ))
5. **Outputs result**: Produces a value between 0 and 1 (for sigmoid)

### From Our Previous Lessons

In lessons 1A and 1B, we built **logistic regression** - which is actually a single artificial neuron:

```
Cell measurements → Weighted sum → Sigmoid → Probability of cancer
    (inputs)         (z = wx + b)    σ(z)        (output)
```

This single neuron achieved 97% accuracy by learning optimal weights for each feature.

In lessons 2A and 2B, we explored **decision trees** - a completely different approach:
- Trees make sequences of if-then decisions
- Each split creates hard boundaries in feature space
- No weights or gradients - just recursive partitioning
- Great for capturing non-linear patterns and interactions

### Why We Need Neural Networks

Both approaches have limitations:

**Logistic Regression (Single Neuron)**:
- ✓ Smooth probability outputs
- ✓ Well-calibrated confidence scores
- ✗ Only learns linear decision boundaries
- ✗ Can't capture feature interactions

**Decision Trees**:
- ✓ Captures non-linear patterns naturally
- ✓ Handles feature interactions well
- ✗ Makes hard splits (not smooth)
- ✗ Can overfit with deep trees

What if we could combine the best of both worlds?

### Enter Neural Networks

Neural networks achieve this by connecting multiple neurons in layers:
- Like logistic regression: smooth, differentiable computations
- Like decision trees: can learn non-linear patterns
- Unlike both: learns hierarchical representations

Think of it like assembling a team of specialist doctors:
- One expert notices subtle cell boundaries
- Another recognises texture patterns
- A third identifies size abnormalities
- Together, they make better diagnoses than any individual

Here's the magic: by combining multiple simple neurons, each learning different patterns, we can solve complex problems that neither logistic regression nor decision trees handle well alone.

For example, imagine diagnosing cancer where:
- Neuron 1 learns: "Are cells large?"
- Neuron 2 learns: "Are cells irregular?"
- Neuron 3 combines their outputs: "Large AND irregular → high risk"

This creates non-linear decision boundaries - something a single neuron could never achieve!

### What We'll Learn

In this lesson, we'll build from our single-neuron foundation to understand:
1. How neurons connect to form networks
2. The mathematics of forward and backward propagation
3. Why activation functions enable non-linear learning
4. How to design network architectures
5. Techniques for training deep networks

By the end, you'll understand not just how neural networks work, but why they're so powerful for pattern recognition in medical diagnosis and beyond. You'll see them not as something mysteriously complex, but as a natural extension of the simple neuron (logistic regression) we've already mastered - one that achieves the flexibility of decision trees while maintaining smooth, probabilistic outputs.

## Table of Contents

1. [Introduction](#introduction)
2. [Required Libraries](#required-libraries)
3. [From Single Neuron to Network](#from-single-neuron-to-network)
   - [Recap: The Logistic Regression Neuron](#recap-the-logistic-regression-neuron)
   - [The Limitation of Single Neurons](#the-limitation-of-single-neurons)
   - [The Power of Combining Neurons](#the-power-of-combining-neurons)
4. [Anatomy of a Neural Network](#anatomy-of-a-neural-network)
   - [Layers and Architecture](#layers-and-architecture)
   - [Connections and Weights](#connections-and-weights)
   - [The Universal Approximation Theorem](#the-universal-approximation-theorem)
5. [Activation Functions: Adding Non-linearity](#activation-functions-adding-non-linearity)
   - [Why We Need Activation Functions](#why-we-need-activation-functions)
   - [Common Activation Functions](#common-activation-functions)
   - [Choosing the Right Activation](#choosing-the-right-activation)
6. [Forward Propagation: Making Predictions](#forward-propagation-making-predictions)
   - [Layer-by-Layer Computation](#layer-by-layer-computation)
   - [Matrix Operations](#matrix-operations)
   - [A Complete Forward Pass Example](#a-complete-forward-pass-example)
7. [The Loss Function for Neural Networks](#the-loss-function-for-neural-networks)
   - [Cross-Entropy for Classification](#cross-entropy-for-classification)
   - [Mean Squared Error for Regression](#mean-squared-error-for-regression)
8. [Backpropagation: Learning from Mistakes](#backpropagation-learning-from-mistakes)
   - [The Chain Rule Revisited](#the-chain-rule-revisited)
   - [Computing Gradients Layer by Layer](#computing-gradients-layer-by-layer)
   - [The Backpropagation Algorithm](#the-backpropagation-algorithm)
9. [Training Neural Networks](#training-neural-networks)
   - [Gradient Descent and Its Variants](#gradient-descent-and-its-variants)
   - [Learning Rate Scheduling](#learning-rate-scheduling)
   - [Batch Processing](#batch-processing)
10. [Preventing Overfitting](#preventing-overfitting)
    - [Regularization Techniques](#regularization-techniques)
    - [Dropout](#dropout)
    - [Early Stopping](#early-stopping)
11. [Network Architecture Design](#network-architecture-design)
    - [How Deep Should We Go?](#how-deep-should-we-go)
    - [How Wide Should Layers Be?](#how-wide-should-layers-be)
    - [Architecture Patterns](#architecture-patterns)
12. [Conclusion](#conclusion)
    - [Key Takeaways](#key-takeaways)
    - [Looking Ahead to Lesson 3B](#looking-ahead-to-lesson-3b)
    - [Further Reading](#further-reading)

<a name="required-libraries"></a>
## Required Libraries

Let's import the libraries we'll use for visualisations and demonstrations:

| Library | Purpose |
|---------|--------|
| NumPy | Numerical computations |
| Matplotlib | Plotting and visualisations |
| Seaborn | Statistical visualisations |
| IPython | Interactive displays |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import HTML
from matplotlib.patches import Circle, Rectangle, FancyBboxPatch
from matplotlib.patches import ConnectionPatch
import matplotlib.patches as mpatches

# Set random seed for reproducibility
np.random.seed(42)

# Configure plotting
plt.style.use('seaborn-v0_8')
%matplotlib inline

print("Libraries loaded successfully!")

<a name="from-single-neuron-to-network"></a>
## From Single Neuron to Network

<a name="recap-the-logistic-regression-neuron"></a>
### Recap: The Logistic Regression Neuron

In Lesson 1A, we built a single neuron that performed logistic regression:

```
Input Features → Linear Combination → Sigmoid → Prediction
     (x)           (z = wx + b)       σ(z)        (0 or 1)
```

This single neuron learned to combine features linearly:
- Cell size × weight₁
- Cell shape × weight₂
- Sum them up with bias
- Apply sigmoid for probability

It achieved 97% accuracy, but there's a fundamental limitation...

<a name="the-limitation-of-single-neurons"></a>
### The Limitation of Single Neurons

A single neuron can only learn linear decision boundaries. Let's visualise this limitation:

In [ ]:
# Demonstrate XOR problem - the classic example of linear inseparability
def plot_xor_problem():
    # XOR data points
    X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
    y = np.array([0, 1, 1, 0])  # XOR output
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    # Plot 1: XOR problem
    colors = ['blue' if label == 0 else 'red' for label in y]
    ax1.scatter(X[:, 0], X[:, 1], c=colors, s=200, edgecolors='black', linewidth=2)
    
    # Try to draw a single line to separate them
    x_line = np.linspace(-0.5, 1.5, 100)
    y_line = 1 - x_line  # Best attempt at linear separation
    ax1.plot(x_line, y_line, 'k--', label='Best Linear Boundary')
    
    ax1.set_xlim(-0.5, 1.5)
    ax1.set_ylim(-0.5, 1.5)
    ax1.set_xlabel('Feature 1')
    ax1.set_ylabel('Feature 2')
    ax1.set_title('XOR Problem: Impossible for Single Neuron')
    ax1.grid(True, alpha=0.3)
    ax1.legend()
    
    # Add annotations
    ax1.annotate('Class 0', xy=(0, 0), xytext=(-0.3, -0.3),
                arrowprops=dict(arrowstyle='->', color='blue'))
    ax1.annotate('Class 1', xy=(0, 1), xytext=(-0.3, 1.3),
                arrowprops=dict(arrowstyle='->', color='red'))
    
    # Plot 2: What we need - non-linear boundary
    xx, yy = np.meshgrid(np.linspace(-0.5, 1.5, 100),
                         np.linspace(-0.5, 1.5, 100))
    
    # Create XOR-like decision boundary
    Z = np.logical_xor(xx > 0.5, yy > 0.5).astype(float)
    
    ax2.contourf(xx, yy, Z, levels=[0, 0.5, 1], colors=['lightblue', 'lightcoral'], alpha=0.6)
    ax2.scatter(X[:, 0], X[:, 1], c=colors, s=200, edgecolors='black', linewidth=2)
    
    ax2.set_xlim(-0.5, 1.5)
    ax2.set_ylim(-0.5, 1.5)
    ax2.set_xlabel('Feature 1')
    ax2.set_ylabel('Feature 2')
    ax2.set_title('What We Need: Non-linear Decision Boundary')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

plot_xor_problem()

The XOR problem shows why we need neural networks:
- Points at (0,0) and (1,1) belong to class 0 (blue)
- Points at (0,1) and (1,0) belong to class 1 (red)
- No single straight line can separate these classes!

<a name="the-power-of-combining-neurons"></a>
### The Power of Combining Neurons

The breakthrough: combine multiple neurons to create non-linear decision boundaries. Here's how:

In [ ]:
def visualise_neuron_combination():
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # XOR data
    X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
    y = np.array([0, 1, 1, 0])
    
    # Hidden neuron 1: Detects upper-left region
    def neuron1(x1, x2):
        return 1 / (1 + np.exp(-(x1 + x2 - 0.5)))
    
    # Hidden neuron 2: Detects lower-right region  
    def neuron2(x1, x2):
        return 1 / (1 + np.exp(-(-x1 - x2 + 1.5)))
    
    # Create mesh grid
    xx, yy = np.meshgrid(np.linspace(-0.5, 1.5, 100),
                         np.linspace(-0.5, 1.5, 100))
    
    # Plot hidden neuron 1
    Z1 = neuron1(xx, yy)
    im1 = axes[0, 0].contourf(xx, yy, Z1, levels=20, cmap='RdBu_r')
    axes[0, 0].scatter(X[:, 0], X[:, 1], c=['black']*4, s=100, edgecolors='white', linewidth=2)
    axes[0, 0].set_title('Hidden Neuron 1\n"Is x₁ + x₂ > 0.5?"')
    axes[0, 0].set_xlabel('x₁')
    axes[0, 0].set_ylabel('x₂')
    plt.colorbar(im1, ax=axes[0, 0])
    
    # Plot hidden neuron 2
    Z2 = neuron2(xx, yy)
    im2 = axes[0, 1].contourf(xx, yy, Z2, levels=20, cmap='RdBu_r')
    axes[0, 1].scatter(X[:, 0], X[:, 1], c=['black']*4, s=100, edgecolors='white', linewidth=2)
    axes[0, 1].set_title('Hidden Neuron 2\n"Is x₁ + x₂ < 1.5?"')
    axes[0, 1].set_xlabel('x₁')
    axes[0, 1].set_ylabel('x₂')
    plt.colorbar(im2, ax=axes[0, 1])
    
    # Plot network architecture
    ax_arch = axes[0, 2]
    ax_arch.set_xlim(0, 10)
    ax_arch.set_ylim(0, 10)
    ax_arch.axis('off')
    
    # Input layer
    ax_arch.add_patch(Circle((2, 7), 0.5, color='lightgreen'))
    ax_arch.add_patch(Circle((2, 3), 0.5, color='lightgreen'))
    ax_arch.text(2, 7, 'x₁', ha='center', va='center')
    ax_arch.text(2, 3, 'x₂', ha='center', va='center')
    
    # Hidden layer
    ax_arch.add_patch(Circle((5, 7), 0.5, color='lightblue'))
    ax_arch.add_patch(Circle((5, 3), 0.5, color='lightblue'))
    ax_arch.text(5, 7, 'h₁', ha='center', va='center')
    ax_arch.text(5, 3, 'h₂', ha='center', va='center')
    
    # Output layer
    ax_arch.add_patch(Circle((8, 5), 0.5, color='lightcoral'))
    ax_arch.text(8, 5, 'y', ha='center', va='center')
    
    # Connections
    for start_y in [7, 3]:
        for end_y in [7, 3]:
            ax_arch.plot([2.5, 4.5], [start_y, end_y], 'k-', alpha=0.3)
    for start_y in [7, 3]:
        ax_arch.plot([5.5, 7.5], [start_y, 5], 'k-', alpha=0.3)
    
    ax_arch.set_title('Neural Network Architecture')
    
    # Show what each neuron outputs for XOR inputs
    axes[1, 0].axis('off')
    output_text = "Hidden Neuron Outputs:\n\n"
    output_text += "Input | h₁  | h₂\n"
    output_text += "-" * 20 + "\n"
    for i, (x1, x2) in enumerate(X):
        h1 = neuron1(x1, x2)
        h2 = neuron2(x1, x2)
        output_text += f"({x1},{x2}) | {h1:.2f} | {h2:.2f}\n"
    axes[1, 0].text(0.1, 0.5, output_text, fontsize=12, family='monospace',
                    transform=axes[1, 0].transAxes)
    
    # Output neuron combination
    def output_neuron(h1, h2):
        # XOR logic: high when exactly one hidden neuron is active
        return 1 / (1 + np.exp(-(20*h1 - 20*h2)))
    
    H1 = neuron1(xx, yy)
    H2 = neuron2(xx, yy)
    Z_final = output_neuron(H1, H2)
    
    im3 = axes[1, 1].contourf(xx, yy, Z_final, levels=20, cmap='RdBu_r')
    colors = ['blue' if label == 0 else 'red' for label in y]
    axes[1, 1].scatter(X[:, 0], X[:, 1], c=colors, s=200, edgecolors='black', linewidth=2)
    axes[1, 1].set_title('Final Output: XOR Solved!')
    axes[1, 1].set_xlabel('x₁')
    axes[1, 1].set_ylabel('x₂')
    plt.colorbar(im3, ax=axes[1, 1])
    
    # Final predictions
    axes[1, 2].axis('off')
    pred_text = "Final Predictions:\n\n"
    pred_text += "Input | Target | Predicted\n"
    pred_text += "-" * 30 + "\n"
    for i, (x1, x2) in enumerate(X):
        h1 = neuron1(x1, x2)
        h2 = neuron2(x1, x2)
        pred = output_neuron(h1, h2)
        pred_text += f"({x1},{x2}) |   {y[i]}    |   {pred:.3f}\n"
    axes[1, 2].text(0.1, 0.5, pred_text, fontsize=12, family='monospace',
                    transform=axes[1, 2].transAxes)
    
    plt.tight_layout()
    plt.show()

visualise_neuron_combination()

This demonstrates the key insight:
1. **Hidden neurons learn features**: Each hidden neuron detects a different pattern
2. **Output neuron combines features**: The final neuron combines these patterns
3. **Non-linear boundaries emerge**: Together, they solve problems impossible for single neurons

This is the foundation of deep learning - simple units combining to solve complex problems!

<a name="anatomy-of-a-neural-network"></a>
## Anatomy of a Neural Network

<a name="layers-and-architecture"></a>
### Layers and Architecture

A neural network consists of layers of interconnected neurons:

1. **Input Layer**: Receives raw features (not true neurons, just data)
2. **Hidden Layers**: Process and transform information
3. **Output Layer**: Produces final predictions

Let's visualise a more complex network:

In [ ]:
def draw_neural_network():
    fig, ax = plt.subplots(figsize=(12, 8))
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 10)
    ax.axis('off')
    
    # Layer positions
    layers = {
        'input': {'x': 1, 'neurons': 4, 'color': 'lightgreen'},
        'hidden1': {'x': 3.5, 'neurons': 5, 'color': 'lightblue'},
        'hidden2': {'x': 6, 'neurons': 3, 'color': 'lightblue'},
        'output': {'x': 8.5, 'neurons': 2, 'color': 'lightcoral'}
    }
    
    # Draw neurons
    neuron_positions = {}
    for layer_name, layer_info in layers.items():
        x = layer_info['x']
        n_neurons = layer_info['neurons']
        color = layer_info['color']
        
        # Calculate y positions to center neurons
        y_start = 5 - (n_neurons - 1) * 0.8
        
        neuron_positions[layer_name] = []
        for i in range(n_neurons):
            y = y_start + i * 1.6
            circle = Circle((x, y), 0.3, color=color, ec='black', linewidth=2)
            ax.add_patch(circle)
            neuron_positions[layer_name].append((x, y))
            
            # Add labels
            if layer_name == 'input':
                ax.text(x-0.7, y, f'x_{i+1}', fontsize=10, ha='right', va='center')
            elif layer_name == 'output':
                ax.text(x+0.7, y, f'y_{i+1}', fontsize=10, ha='left', va='center')
    
    # Draw connections
    layer_pairs = [('input', 'hidden1'), ('hidden1', 'hidden2'), ('hidden2', 'output')]
    
    for layer1, layer2 in layer_pairs:
        for start_pos in neuron_positions[layer1]:
            for end_pos in neuron_positions[layer2]:
                # Random weight for visualization
                weight = np.random.uniform(-1, 1)
                color = 'blue' if weight > 0 else 'red'
                alpha = min(abs(weight), 0.5)
                
                ax.plot([start_pos[0] + 0.3, end_pos[0] - 0.3],
                       [start_pos[1], end_pos[1]],
                       color=color, alpha=alpha, linewidth=1)
    
    # Add layer labels
    ax.text(1, 9, 'Input\nLayer', fontsize=12, ha='center', weight='bold')
    ax.text(3.5, 9, 'Hidden\nLayer 1', fontsize=12, ha='center', weight='bold')
    ax.text(6, 9, 'Hidden\nLayer 2', fontsize=12, ha='center', weight='bold')
    ax.text(8.5, 9, 'Output\nLayer', fontsize=12, ha='center', weight='bold')
    
    # Add annotations
    ax.text(5, 0.5, 'Blue connections: positive weights\nRed connections: negative weights\nOpacity: weight magnitude',
            fontsize=10, ha='center', style='italic')
    
    plt.title('Multi-Layer Neural Network Architecture', fontsize=16, pad=20)
    plt.tight_layout()
    plt.show()

draw_neural_network()

<a name="connections-and-weights"></a>
### Connections and Weights

Each connection between neurons has a weight that determines its strength:
- **Positive weights** (blue): Excitatory connections that increase activation
- **Negative weights** (red): Inhibitory connections that decrease activation
- **Weight magnitude**: Determines influence strength

For a network with:
- 4 input features
- 5 neurons in hidden layer 1
- 3 neurons in hidden layer 2
- 2 output neurons

We need:
- Input → Hidden1: 4 × 5 = 20 weights + 5 biases
- Hidden1 → Hidden2: 5 × 3 = 15 weights + 3 biases
- Hidden2 → Output: 3 × 2 = 6 weights + 2 biases
- **Total: 51 parameters to learn!**

<a name="the-universal-approximation-theorem"></a>
### The Universal Approximation Theorem

A remarkable mathematical result states:
> A neural network with just one hidden layer containing enough neurons can approximate any continuous function to arbitrary accuracy.

Let's visualise this power:

In [ ]:
def demonstrate_universal_approximation():
    # Create a complex target function
    x = np.linspace(-2, 2, 1000)
    y_target = np.sin(2*x) + 0.5*np.cos(5*x) + 0.2*x
    
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    
    # Different numbers of hidden neurons
    neuron_counts = [2, 5, 10, 20]
    
    for idx, n_neurons in enumerate(neuron_counts):
        ax = axes[idx // 2, idx % 2]
        
        # Plot target function
        ax.plot(x, y_target, 'k-', label='Target Function', linewidth=2)
        
        # Simulate neural network approximation with ReLU-like basis functions
        y_approx = np.zeros_like(x)
        
        for i in range(n_neurons):
            # Random initialization for demonstration
            center = np.random.uniform(-2, 2)
            weight = np.random.uniform(-1, 1)
            width = np.random.uniform(0.5, 2)
            
            # ReLU-like activation
            activation = np.maximum(0, (x - center) / width)
            y_approx += weight * activation
            
            # Plot individual neuron contribution
            ax.plot(x, weight * activation, '--', alpha=0.3, linewidth=1)
        
        # Normalize to match target scale
        y_approx = (y_approx - y_approx.mean()) / y_approx.std() * y_target.std() + y_target.mean()
        
        # Plot approximation
        ax.plot(x, y_approx, 'r-', label=f'NN Approximation', linewidth=2)
        
        ax.set_title(f'Network with {n_neurons} Hidden Neurons')
        ax.set_xlabel('Input')
        ax.set_ylabel('Output')
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        # Calculate and display error
        mse = np.mean((y_target - y_approx)**2)
        ax.text(0.02, 0.98, f'MSE: {mse:.3f}', transform=ax.transAxes,
                verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.suptitle('Universal Approximation: More Neurons = Better Approximation', fontsize=16)
    plt.tight_layout()
    plt.show()

demonstrate_universal_approximation()

Key insights:
1. **Few neurons**: Rough approximation
2. **More neurons**: Better fit to complex functions
3. **Each neuron**: Contributes a "basis function"
4. **Together**: Can approximate any shape!

But if one hidden layer is enough, why go deeper? Deep networks can be more efficient, learning hierarchical representations with fewer total parameters.

<a name="activation-functions-adding-non-linearity"></a>
## Activation Functions: Adding Non-linearity

<a name="why-we-need-activation-functions"></a>
### Why We Need Activation Functions

Without activation functions, even a 100-layer network would be equivalent to a single linear transformation:

```
Linear(Linear(Linear(x))) = Linear(x)
```

Activation functions add the non-linearity that gives neural networks their power.

<a name="common-activation-functions"></a>
### Common Activation Functions

Let's explore the most important activation functions:

In [ ]:
def plot_activation_functions():
    x = np.linspace(-3, 3, 1000)
    
    # Define activation functions
    def sigmoid(x):
        return 1 / (1 + np.exp(-x))
    
    def tanh(x):
        return np.tanh(x)
    
    def relu(x):
        return np.maximum(0, x)
    
    def leaky_relu(x, alpha=0.1):
        return np.where(x > 0, x, alpha * x)
    
    # Create subplots
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    
    # Sigmoid
    ax = axes[0, 0]
    ax.plot(x, sigmoid(x), 'b-', linewidth=2)
    ax.set_title('Sigmoid: σ(x) = 1/(1+e^(-x))', fontsize=14)
    ax.set_xlabel('x')
    ax.set_ylabel('σ(x)')
    ax.grid(True, alpha=0.3)
    ax.axhline(y=0, color='k', linewidth=0.5)
    ax.axvline(x=0, color='k', linewidth=0.5)
    ax.set_ylim(-0.1, 1.1)
    
    # Add annotations
    ax.annotate('Saturates at 1', xy=(2, 0.88), xytext=(1, 0.7),
                arrowprops=dict(arrowstyle='->', color='red'))
    ax.annotate('Saturates at 0', xy=(-2, 0.12), xytext=(-1, 0.3),
                arrowprops=dict(arrowstyle='->', color='red'))
    
    # Tanh
    ax = axes[0, 1]
    ax.plot(x, tanh(x), 'g-', linewidth=2)
    ax.set_title('Tanh: tanh(x) = (e^x - e^(-x))/(e^x + e^(-x))', fontsize=14)
    ax.set_xlabel('x')
    ax.set_ylabel('tanh(x)')
    ax.grid(True, alpha=0.3)
    ax.axhline(y=0, color='k', linewidth=0.5)
    ax.axvline(x=0, color='k', linewidth=0.5)
    ax.set_ylim(-1.1, 1.1)
    
    # ReLU
    ax = axes[1, 0]
    ax.plot(x, relu(x), 'r-', linewidth=2)
    ax.set_title('ReLU: f(x) = max(0, x)', fontsize=14)
    ax.set_xlabel('x')
    ax.set_ylabel('ReLU(x)')
    ax.grid(True, alpha=0.3)
    ax.axhline(y=0, color='k', linewidth=0.5)
    ax.axvline(x=0, color='k', linewidth=0.5)
    ax.set_ylim(-0.5, 3)
    
    # Add annotation
    ax.annotate('Dead for x < 0', xy=(-1.5, 0), xytext=(-2, 1),
                arrowprops=dict(arrowstyle='->', color='red'))
    
    # Leaky ReLU
    ax = axes[1, 1]
    ax.plot(x, leaky_relu(x), 'm-', linewidth=2)
    ax.set_title('Leaky ReLU: f(x) = max(0.1x, x)', fontsize=14)
    ax.set_xlabel('x')
    ax.set_ylabel('Leaky ReLU(x)')
    ax.grid(True, alpha=0.3)
    ax.axhline(y=0, color='k', linewidth=0.5)
    ax.axvline(x=0, color='k', linewidth=0.5)
    ax.set_ylim(-0.5, 3)
    
    # Add annotation
    ax.annotate('Small gradient\nfor x < 0', xy=(-2, -0.2), xytext=(-1.5, -1),
                arrowprops=dict(arrowstyle='->', color='green'))
    
    plt.suptitle('Common Activation Functions', fontsize=16)
    plt.tight_layout()
    plt.show()

plot_activation_functions()

### Comparing Activation Functions

| Function | Range | Pros | Cons | Best Use |
|----------|-------|------|------|----------|
| Sigmoid | (0,1) | Smooth, probabilistic interpretation | Vanishing gradients, not zero-centered | Output layer for binary classification |
| Tanh | (-1,1) | Zero-centered, smooth | Vanishing gradients | Hidden layers in RNNs |
| ReLU | [0,∞) | No vanishing gradients, fast | Dead neurons, not zero-centered | Default for hidden layers |
| Leaky ReLU | (-∞,∞) | Avoids dead neurons | Less common, extra hyperparameter | When ReLU causes issues |

<a name="choosing-the-right-activation"></a>
### Choosing the Right Activation

Let's see how activation choice affects learning:

In [ ]:
def compare_activation_gradients():
    x = np.linspace(-3, 3, 1000)
    
    # Gradients of activation functions
    def sigmoid_grad(x):
        s = 1 / (1 + np.exp(-x))
        return s * (1 - s)
    
    def tanh_grad(x):
        return 1 - np.tanh(x)**2
    
    def relu_grad(x):
        return (x > 0).astype(float)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    ax.plot(x, sigmoid_grad(x), 'b-', label='Sigmoid gradient', linewidth=2)
    ax.plot(x, tanh_grad(x), 'g-', label='Tanh gradient', linewidth=2)
    ax.plot(x, relu_grad(x), 'r-', label='ReLU gradient', linewidth=2)
    
    ax.set_xlabel('x', fontsize=12)
    ax.set_ylabel('Gradient', fontsize=12)
    ax.set_title('Gradients of Activation Functions', fontsize=14)
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Highlight vanishing gradient regions
    ax.axhspan(0, 0.1, xmin=0, xmax=0.2, alpha=0.2, color='red')
    ax.axhspan(0, 0.1, xmin=0.8, xmax=1, alpha=0.2, color='red')
    ax.text(-2.5, 0.05, 'Vanishing\ngradient', color='red', fontsize=10)
    ax.text(2, 0.05, 'Vanishing\ngradient', color='red', fontsize=10)
    
    plt.tight_layout()
    plt.show()

compare_activation_gradients()

Key insights:
1. **Sigmoid/Tanh**: Gradients vanish for large inputs → slow learning
2. **ReLU**: Constant gradient for positive inputs → faster learning
3. **Trade-off**: ReLU is simple but can "die" (output zero forever)

This is why ReLU became the default for deep networks!

<a name="forward-propagation-making-predictions"></a>
## Forward Propagation: Making Predictions

Forward propagation is how a neural network processes input to produce output. Let's trace through this step by step.

<a name="layer-by-layer-computation"></a>
### Layer-by-Layer Computation

For each layer, we:
1. Take inputs from previous layer
2. Multiply by weights and add bias
3. Apply activation function
4. Pass to next layer

Mathematically:
```
z = Wx + b        # Linear transformation
a = f(z)          # Activation
```

<a name="matrix-operations"></a>
### Matrix Operations

Let's see how this works with real numbers:

In [ ]:
def forward_propagation_example():
    # Example: 2 inputs → 3 hidden → 1 output
    np.random.seed(42)
    
    # Input
    x = np.array([[0.5], [0.8]])  # 2x1 vector
    
    # Layer 1: Input (2) → Hidden (3)
    W1 = np.array([[0.2, -0.3],    # 3x2 matrix
                   [0.4, 0.1],
                   [-0.5, 0.6]])
    b1 = np.array([[0.1], [0.2], [-0.1]])  # 3x1 vector
    
    # Forward pass through layer 1
    z1 = W1 @ x + b1  # @ is matrix multiplication
    a1 = np.maximum(0, z1)  # ReLU activation
    
    # Layer 2: Hidden (3) → Output (1)
    W2 = np.array([[0.3, -0.2, 0.5]])  # 1x3 matrix
    b2 = np.array([[0.1]])  # 1x1 vector
    
    # Forward pass through layer 2
    z2 = W2 @ a1 + b2
    a2 = 1 / (1 + np.exp(-z2))  # Sigmoid for output
    
    # Visualize the computation
    fig, axes = plt.subplots(1, 3, figsize=(15, 6))
    
    # Plot 1: Layer 1 computation
    ax = axes[0]
    ax.text(0.5, 0.9, 'Layer 1: Input → Hidden', ha='center', fontsize=14, weight='bold')
    
    # Show matrix multiplication
    computation_text = f"""Input x:
[{x[0,0]:.1f}]
[{x[1,0]:.1f}]

Weights W1:
[{W1[0,0]:>4.1f} {W1[0,1]:>4.1f}]
[{W1[1,0]:>4.1f} {W1[1,1]:>4.1f}]
[{W1[2,0]:>4.1f} {W1[2,1]:>4.1f}]

Bias b1:
[{b1[0,0]:>4.1f}]
[{b1[1,0]:>4.1f}]
[{b1[2,0]:>4.1f}]

z1 = W1·x + b1:
[{z1[0,0]:>4.2f}]
[{z1[1,0]:>4.2f}]
[{z1[2,0]:>4.2f}]

a1 = ReLU(z1):
[{a1[0,0]:>4.2f}]
[{a1[1,0]:>4.2f}]
[{a1[2,0]:>4.2f}]"""
    
    ax.text(0.1, 0.5, computation_text, fontsize=10, family='monospace',
            verticalalignment='center')
    ax.axis('off')
    
    # Plot 2: Activation visualization
    ax = axes[1]
    neurons_x = [0.2, 0.5, 0.8]
    
    # Draw neurons with values
    for i, (nx, z_val, a_val) in enumerate(zip(neurons_x, z1.flatten(), a1.flatten())):
        # Draw neuron
        color = 'lightgreen' if a_val > 0 else 'lightcoral'
        circle = plt.Circle((nx, 0.5), 0.1, color=color, ec='black', linewidth=2)
        ax.add_patch(circle)
        
        # Show values
        ax.text(nx, 0.8, f'z = {z_val:.2f}', ha='center', fontsize=10)
        ax.text(nx, 0.2, f'a = {a_val:.2f}', ha='center', fontsize=10)
        ax.text(nx, 0.5, f'h{i+1}', ha='center', va='center', fontsize=12, weight='bold')
    
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_title('Hidden Layer Activations', fontsize=14)
    ax.axis('off')
    
    # Plot 3: Layer 2 computation
    ax = axes[2]
    ax.text(0.5, 0.9, 'Layer 2: Hidden → Output', ha='center', fontsize=14, weight='bold')
    
    computation_text2 = f"""Hidden activations a1:
[{a1[0,0]:.2f}]
[{a1[1,0]:.2f}]
[{a1[2,0]:.2f}]

Weights W2:
[{W2[0,0]:>4.1f} {W2[0,1]:>4.1f} {W2[0,2]:>4.1f}]

Bias b2: [{b2[0,0]:.1f}]

z2 = W2·a1 + b2 = {z2[0,0]:.3f}

Output = σ(z2) = {a2[0,0]:.3f}

Probability: {a2[0,0]*100:.1f}%"""
    
    ax.text(0.1, 0.5, computation_text2, fontsize=11, family='monospace',
            verticalalignment='center')
    ax.axis('off')
    
    plt.suptitle('Forward Propagation: Step-by-Step Computation', fontsize=16)
    plt.tight_layout()
    plt.show()
    
forward_propagation_example()

<a name="a-complete-forward-pass-example"></a>
### A Complete Forward Pass Example

Let's implement forward propagation for a simple medical diagnosis network:

In [ ]:
class SimpleNeuralNetwork:
    """A simple 2-layer neural network for demonstration"""
    
    def __init__(self, input_size, hidden_size, output_size):
        # Initialize weights with small random values
        self.W1 = np.random.randn(hidden_size, input_size) * 0.1
        self.b1 = np.zeros((hidden_size, 1))
        self.W2 = np.random.randn(output_size, hidden_size) * 0.1
        self.b2 = np.zeros((output_size, 1))
    
    def relu(self, z):
        """ReLU activation function"""
        return np.maximum(0, z)
    
    def sigmoid(self, z):
        """Sigmoid activation function"""
        return 1 / (1 + np.exp(-z))
    
    def forward(self, X):
        """Forward propagation through the network"""
        # Layer 1
        self.z1 = self.W1 @ X + self.b1
        self.a1 = self.relu(self.z1)
        
        # Layer 2
        self.z2 = self.W2 @ self.a1 + self.b2
        self.a2 = self.sigmoid(self.z2)
        
        return self.a2

# Create and test the network
nn = SimpleNeuralNetwork(input_size=4, hidden_size=5, output_size=1)

# Example: 4 medical features (normalized)
# [cell_size, cell_shape, cell_texture, cell_symmetry]
patient_features = np.array([[1.2], [-0.3], [0.8], [0.5]])

# Forward pass
prediction = nn.forward(patient_features)

print("Medical Diagnosis Neural Network")
print("=" * 40)
print(f"Input features shape: {patient_features.shape}")
print(f"Hidden layer neurons: {nn.W1.shape[0]}")
print(f"Output neurons: {nn.W2.shape[0]}")
print(f"\nPatient features:")
feature_names = ['Cell Size', 'Cell Shape', 'Cell Texture', 'Cell Symmetry']
for name, value in zip(feature_names, patient_features.flatten()):
    print(f"  {name}: {value:>6.2f}")
print(f"\nDiagnosis probability: {prediction[0,0]:.3f}")
print(f"Prediction: {'Malignant' if prediction[0,0] > 0.5 else 'Benign'}")

Key insights from forward propagation:
1. **Linear transformations**: Each layer applies Wx + b
2. **Non-linear activations**: Enable complex decision boundaries
3. **Compositional**: Each layer builds on the previous
4. **Differentiable**: Enables backpropagation for learning

<a name="the-loss-function-for-neural-networks"></a>
## The Loss Function for Neural Networks

Just like logistic regression, neural networks need a loss function to measure prediction quality.

<a name="cross-entropy-for-classification"></a>
### Cross-Entropy for Classification

For binary classification (like cancer detection):
$$L = -\frac{1}{n}\sum_{i=1}^{n}[y_i \log(\hat{y}_i) + (1-y_i)\log(1-\hat{y}_i)]$$

For multi-class classification:
$$L = -\frac{1}{n}\sum_{i=1}^{n}\sum_{j=1}^{c} y_{ij} \log(\hat{y}_{ij})$$

where c is the number of classes.

<a name="mean-squared-error-for-regression"></a>
### Mean Squared Error for Regression

For continuous outputs:
$$L = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2$$

Let's visualise how loss changes during training:

In [ ]:
def visualise_loss_landscape()

The loss landscape shows:
1. **Multiple local minima**: Neural networks have complex loss surfaces
2. **Gradient descent path**: How optimization navigates the landscape
3. **Challenge**: Finding the global minimum among many local ones

<a name="backpropagation-learning-from-mistakes"></a>
## Backpropagation: Learning from Mistakes

Backpropagation is how neural networks learn. It efficiently computes gradients for all weights by propagating errors backward through the network.

<a name="the-chain-rule-revisited"></a>
### The Chain Rule Revisited

Remember from Lesson 1A, the chain rule tells us:
$\frac{\partial L}{\partial w} = \frac{\partial L}{\partial a} \cdot \frac{\partial a}{\partial z} \cdot \frac{\partial z}{\partial w}$

For neural networks, we apply this layer by layer, backward from output to input.

<a name="computing-gradients-layer-by-layer"></a>
### Computing Gradients Layer by Layer

Let's visualise the backpropagation process:

In [ ]:
def visualise_backpropagation():
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Network architecture for visualization
    ax = axes[0, 0]
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 10)
    ax.axis('off')
    ax.set_title('Network Architecture', fontsize=14, weight='bold')
    
    # Draw network
    # Input layer
    input_neurons = [(2, 7), (2, 5), (2, 3)]
    for i, (x, y) in enumerate(input_neurons):
        ax.add_patch(Circle((x, y), 0.4, color='lightgreen', ec='black'))
        ax.text(x, y, f'x{i+1}', ha='center', va='center')
    
    # Hidden layer
    hidden_neurons = [(5, 6), (5, 4)]
    for i, (x, y) in enumerate(hidden_neurons):
        ax.add_patch(Circle((x, y), 0.4, color='lightblue', ec='black'))
        ax.text(x, y, f'h{i+1}', ha='center', va='center')
    
    # Output
    ax.add_patch(Circle((8, 5), 0.4, color='lightcoral', ec='black'))
    ax.text(8, 5, 'y', ha='center', va='center')
    
    # Connections
    for start in input_neurons:
        for end in hidden_neurons:
            ax.plot([start[0]+0.4, end[0]-0.4], [start[1], end[1]], 'k-', alpha=0.3)
    for start in hidden_neurons:
        ax.plot([start[0]+0.4, 8-0.4], [start[1], 5], 'k-', alpha=0.3)
    
    # Forward pass visualization
    ax = axes[0, 1]
    ax.axis('off')
    ax.set_title('Forward Pass', fontsize=14, weight='bold')
    
    forward_text = """Step 1: Input → Hidden
z₁ = W₁x + b₁
a₁ = ReLU(z₁)

Step 2: Hidden → Output
z₂ = W₂a₁ + b₂
a₂ = σ(z₂)

Step 3: Compute Loss
L = -[y·log(a₂) + (1-y)·log(1-a₂)]

Example values:
x = [0.5, -0.2, 0.8]
a₁ = [0.3, 0.0] (after ReLU)
a₂ = 0.73
y = 1 (true label)
L = 0.31"""
    
    ax.text(0.1, 0.5, forward_text, fontsize=11, family='monospace',
            transform=ax.transAxes, verticalalignment='center')
    
    # Backward pass visualization
    ax = axes[1, 0]
    ax.axis('off')
    ax.set_title('Backward Pass', fontsize=14, weight='bold')
    
    backward_text = """Step 1: Output gradient
∂L/∂a₂ = (a₂ - y)/(a₂(1-a₂))
       = (0.73 - 1)/(0.73·0.27) = -1.37

Step 2: Hidden gradients
∂L/∂z₂ = ∂L/∂a₂ · ∂a₂/∂z₂
       = -1.37 · a₂(1-a₂)
       = -1.37 · 0.197 = -0.27

∂L/∂W₂ = ∂L/∂z₂ · a₁ᵀ
∂L/∂b₂ = ∂L/∂z₂

Step 3: Input layer gradients
∂L/∂a₁ = W₂ᵀ · ∂L/∂z₂
∂L/∂z₁ = ∂L/∂a₁ · ReLU'(z₁)
∂L/∂W₁ = ∂L/∂z₁ · xᵀ
∂L/∂b₁ = ∂L/∂z₁"""
    
    ax.text(0.1, 0.5, backward_text, fontsize=10, family='monospace',
            transform=ax.transAxes, verticalalignment='center')
    
    # Gradient flow visualization
    ax = axes[1, 1]
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 10)
    ax.axis('off')
    ax.set_title('Gradient Flow', fontsize=14, weight='bold')
    
    # Redraw network with gradient flow
    # Output
    ax.add_patch(Circle((8, 5), 0.4, color='red', ec='black', alpha=0.8))
    ax.text(8, 5, '∂L/∂y', ha='center', va='center', fontsize=10)
    
    # Hidden layer with gradients
    for i, (x, y) in enumerate(hidden_neurons):
        ax.add_patch(Circle((x, y), 0.4, color='orange', ec='black', alpha=0.6))
        ax.text(x, y, f'∂L/∂h{i+1}', ha='center', va='center', fontsize=8)
    
    # Input layer
    for i, (x, y) in enumerate(input_neurons):
        ax.add_patch(Circle((x, y), 0.4, color='yellow', ec='black', alpha=0.4))
        ax.text(x, y, f'∂L/∂x{i+1}', ha='center', va='center', fontsize=8)
    
    # Gradient flow arrows
    ax.arrow(7.5, 5, -2, 0, head_width=0.3, head_length=0.2, fc='red', ec='red')
    ax.arrow(7.5, 4.8, -2, 0.8, head_width=0.3, head_length=0.2, fc='red', ec='red')
    
    for hy in [6, 4]:
        ax.arrow(4.5, hy, -2, 0, head_width=0.3, head_length=0.2, fc='orange', ec='orange')
    
    ax.text(5, 1, 'Gradients flow backward through the network', 
            ha='center', fontsize=12, style='italic')
    
    plt.suptitle('Backpropagation: Computing Gradients', fontsize=16)
    plt.tight_layout()
    plt.show()

visualise_backpropagation()

<a name="the-backpropagation-algorithm"></a>
### The Backpropagation Algorithm

Here's a complete implementation of backpropagation:

In [ ]:
class NeuralNetworkWithBackprop:
    """Neural network with full backpropagation implementation"""
    
    def __init__(self, layers):
        """
        Initialize network with given layer sizes.
        layers: list of layer sizes, e.g., [2, 3, 1] for 2→3→1 network
        """
        self.layers = layers
        self.weights = []
        self.biases = []
        
        # Initialize weights and biases for each layer
        for i in range(len(layers) - 1):
            w = np.random.randn(layers[i+1], layers[i]) * 0.1
            b = np.zeros((layers[i+1], 1))
            self.weights.append(w)
            self.biases.append(b)
    
    def relu(self, z):
        return np.maximum(0, z)
    
    def relu_derivative(self, z):
        return (z > 0).astype(float)
    
    def sigmoid(self, z):
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))
    
    def sigmoid_derivative(self, a):
        return a * (1 - a)
    
    def forward(self, X):
        """Forward pass with caching for backprop"""
        self.activations = [X]
        self.z_values = []
        
        A = X
        for i in range(len(self.weights)):
            Z = self.weights[i] @ A + self.biases[i]
            self.z_values.append(Z)
            
            # Use ReLU for hidden layers, sigmoid for output
            if i < len(self.weights) - 1:
                A = self.relu(Z)
            else:
                A = self.sigmoid(Z)
            
            self.activations.append(A)
        
        return A
    
    def backward(self, X, y, learning_rate=0.01):
        """Backpropagation algorithm"""
        m = X.shape[1]  # Number of examples
        
        # Gradients storage
        dW = [None] * len(self.weights)
        db = [None] * len(self.biases)
        
        # Output layer gradient
        dA = self.activations[-1] - y  # For cross-entropy loss
        
        # Backward through layers
        for i in reversed(range(len(self.weights))):
            # Current layer gradients
            if i == len(self.weights) - 1:
                # Output layer (sigmoid)
                dZ = dA  # Simplified for cross-entropy + sigmoid
            else:
                # Hidden layers (ReLU)
                dZ = dA * self.relu_derivative(self.z_values[i])
            
            # Weight and bias gradients
            dW[i] = (1/m) * dZ @ self.activations[i].T
            db[i] = (1/m) * np.sum(dZ, axis=1, keepdims=True)
            
            # Gradient for previous layer
            if i > 0:
                dA = self.weights[i].T @ dZ
        
        # Update weights and biases
        for i in range(len(self.weights)):
            self.weights[i] -= learning_rate * dW[i]
            self.biases[i] -= learning_rate * db[i]
        
        return dW, db
    
    def compute_loss(self, y_true, y_pred):
        """Binary cross-entropy loss"""
        m = y_true.shape[1]
        y_pred = np.clip(y_pred, 1e-15, 1 - 1e-15)
        loss = -(1/m) * np.sum(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))
        return loss

# Demonstrate backpropagation on XOR problem
def demo_backprop_xor():
    # XOR dataset
    X = np.array([[0, 0, 1, 1],
                  [0, 1, 0, 1]])
    y = np.array([[0, 1, 1, 0]])
    
    # Create network: 2 inputs → 3 hidden → 1 output
    nn = NeuralNetworkWithBackprop([2, 3, 1])
    
    # Training
    losses = []
    for epoch in range(1000):
        # Forward pass
        y_pred = nn.forward(X)
        loss = nn.compute_loss(y, y_pred)
        losses.append(loss)
        
        # Backward pass
        nn.backward(X, y, learning_rate=0.5)
        
        if epoch % 100 == 0:
            print(f"Epoch {epoch}, Loss: {loss:.4f}")
    
    # Plot results
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    # Loss curve
    ax1.plot(losses)
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Training Loss')
    ax1.grid(True, alpha=0.3)
    
    # Final predictions
    y_final = nn.forward(X)
    ax2.bar(range(4), y.flatten(), alpha=0.5, label='True')
    ax2.bar(range(4), y_final.flatten(), alpha=0.5, label='Predicted')
    ax2.set_xticks(range(4))
    ax2.set_xticklabels(['(0,0)', '(0,1)', '(1,0)', '(1,1)'])
    ax2.set_ylabel('Output')
    ax2.set_title('XOR Problem: Final Predictions')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\nFinal predictions:")
    for i in range(4):
        print(f"Input: {X[:, i]}, Target: {y[0, i]}, Predicted: {y_final[0, i]:.3f}")

demo_backprop_xor()

Key insights from backpropagation:
1. **Efficient gradient computation**: Reuses forward pass computations
2. **Layer-by-layer**: Works backward from output to input
3. **Chain rule application**: Multiplies gradients through layers
4. **Solves XOR**: The network learns the non-linear pattern!

<a name="training-neural-networks"></a>
## Training Neural Networks

<a name="gradient-descent-and-its-variants"></a>
### Gradient Descent and Its Variants

We've seen basic gradient descent, but modern networks use more sophisticated optimizers:

1. **Stochastic Gradient Descent (SGD)**
   - Updates weights using one sample at a time
   - Noisy but can escape local minima

2. **Mini-batch Gradient Descent**
   - Updates using small batches (e.g., 32 samples)
   - Balances efficiency and stability

3. **Momentum**
   - Adds velocity to updates
   - Helps overcome small local minima

4. **Adam (Adaptive Moment Estimation)**
   - Adapts learning rate for each parameter
   - Combines momentum with adaptive rates

Let's visualise these optimizers:

In [ ]:
def compare_optimizers():
    # Create a simple 2D optimization landscape
    def loss_function(x, y):
        return (1.5 - x + x*y)**2 + (2.25 - x + x*y**2)**2 + (2.625 - x + x*y**3)**2
    
    # Starting point
    start = np.array([0.5, 0.5])
    
    # Optimizer implementations
    def sgd_step(grad, v=None, lr=0.01):
        return -lr * grad, None
    
    def momentum_step(grad, v=None, lr=0.01, beta=0.9):
        if v is None:
            v = np.zeros_like(grad)
        v = beta * v - lr * grad
        return v, v
    
    def adam_step(grad, v=None, lr=0.01, beta1=0.9, beta2=0.999, t=1):
        if v is None:
            m = np.zeros_like(grad)
            v = np.zeros_like(grad)
        else:
            m, v = v
        
        m = beta1 * m + (1 - beta1) * grad
        v = beta2 * v + (1 - beta2) * grad**2
        
        m_hat = m / (1 - beta1**t)
        v_hat = v / (1 - beta2**t)
        
        step = -lr * m_hat / (np.sqrt(v_hat) + 1e-8)
        return step, (m, v)
    
    # Run optimizers
    optimizers = {
        'SGD': (sgd_step, {}),
        'Momentum': (momentum_step, {}),
        'Adam': (adam_step, {})
    }
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    for idx, (name, (step_func, kwargs)) in enumerate(optimizers.items()):
        ax = axes[idx]
        
        # Create contour plot
        x = np.linspace(-1, 4, 100)
        y = np.linspace(-1, 4, 100)
        X, Y = np.meshgrid(x, y)
        Z = loss_function(X, Y)
        
        contour = ax.contour(X, Y, Z, levels=np.logspace(-1, 3, 20))
        ax.clabel(contour, inline=True, fontsize=8)
        
        # Run optimizer
        path = [start.copy()]
        pos = start.copy()
        v = None
        
        for t in range(50):
            # Numerical gradient
            eps = 1e-5
            grad = np.array([
                (loss_function(pos[0] + eps, pos[1]) - loss_function(pos[0] - eps, pos[1])) / (2 * eps),
                (loss_function(pos[0], pos[1] + eps) - loss_function(pos[0], pos[1] - eps)) / (2 * eps)
            ])
            
            # Take step
            if name == 'Adam':
                step, v = step_func(grad, v, t=t+1, **kwargs)
            else:
                step, v = step_func(grad, v, **kwargs)
            
            pos = pos + step
            path.append(pos.copy())
        
        # Plot path
        path = np.array(path)
        ax.plot(path[:, 0], path[:, 1], 'ro-', markersize=4, linewidth=2)
        ax.plot(path[0, 0], path[0, 1], 'go', markersize=10, label='Start')
        ax.plot(path[-1, 0], path[-1, 1], 'r*', markersize=15, label='End')
        
        ax.set_title(f'{name} Optimizer')
        ax.set_xlabel('Parameter 1')
        ax.set_ylabel('Parameter 2')
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    plt.suptitle('Optimizer Comparison', fontsize=16)
    plt.tight_layout()
    plt.show()

compare_optimizers()

<a name="learning-rate-scheduling"></a>
### Learning Rate Scheduling

Learning rate scheduling adjusts the learning rate during training:

In [ ]:
def plot_learning_rate_schedules():
    epochs = np.arange(0, 100)
    initial_lr = 0.1
    
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    
    # Step decay
    ax = axes[0, 0]
    step_lr = initial_lr * (0.5 ** (epochs // 30))
    ax.plot(epochs, step_lr, 'b-', linewidth=2)
    ax.set_title('Step Decay')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Learning Rate')
    ax.grid(True, alpha=0.3)
    
    # Exponential decay
    ax = axes[0, 1]
    exp_lr = initial_lr * np.exp(-0.03 * epochs)
    ax.plot(epochs, exp_lr, 'g-', linewidth=2)
    ax.set_title('Exponential Decay')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Learning Rate')
    ax.grid(True, alpha=0.3)
    
    # Cosine annealing
    ax = axes[1, 0]
    cos_lr = initial_lr * 0.5 * (1 + np.cos(np.pi * epochs / 100))
    ax.plot(epochs, cos_lr, 'r-', linewidth=2)
    ax.set_title('Cosine Annealing')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Learning Rate')
    ax.grid(True, alpha=0.3)
    
    # Warm restart
    ax = axes[1, 1]
    warm_lr = []
    for epoch in epochs:
        cycle = epoch % 20
        lr = initial_lr * 0.5 * (1 + np.cos(np.pi * cycle / 20))
        warm_lr.append(lr)
    ax.plot(epochs, warm_lr, 'm-', linewidth=2)
    ax.set_title('Warm Restart')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Learning Rate')
    ax.grid(True, alpha=0.3)
    
    plt.suptitle('Learning Rate Scheduling Strategies', fontsize=16)
    plt.tight_layout()
    plt.show()

plot_learning_rate_schedules()

<a name="batch-processing"></a>
### Batch Processing

Batch size affects both training dynamics and computational efficiency:

- **Large batches (e.g., 256+)**:
  - More stable gradients
  - Better GPU utilization
  - May converge to sharper minima

- **Small batches (e.g., 32)**:
  - Noisier gradients (regularization effect)
  - More frequent updates
  - Often better generalization

- **Mini-batch (typical: 32-128)**:
  - Balance between stability and noise
  - Good for most applications

<a name="preventing-overfitting"></a>
## Preventing Overfitting

Neural networks can memorize training data. We need techniques to ensure they generalize.

<a name="regularization-techniques"></a>
### Regularization Techniques

1. **L2 Regularization (Weight Decay)**
   $L_{total} = L_{data} + \lambda \sum_i w_i^2$
   
   Penalizes large weights, encouraging simpler models.

2. **L1 Regularization**
   $L_{total} = L_{data} + \lambda \sum_i |w_i|$
   
   Encourages sparsity (many weights become exactly zero).

<a name="dropout"></a>
### Dropout

Dropout randomly "drops" neurons during training:

In [ ]:
def visualize_dropout():
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Original network
    ax = axes[0]
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 10)
    ax.axis('off')
    ax.set_title('Original Network', fontsize=14)
    
    # Draw full network
    layers_pos = [2, 5, 8]
    layer_sizes = [4, 5, 2]
    
    for layer_idx, (x, size) in enumerate(zip(layers_pos, layer_sizes)):
        y_start = 5 - (size - 1) * 0.8
        for i in range(size):
            y = y_start + i * 1.6
            color = ['lightgreen', 'lightblue', 'lightcoral'][layer_idx]
            ax.add_patch(Circle((x, y), 0.3, color=color, ec='black'))
    
    # Draw connections
    for l1, l2, s1, s2 in zip(layers_pos[:-1], layers_pos[1:], layer_sizes[:-1], layer_sizes[1:]):
        y1_start = 5 - (s1 - 1) * 0.8
        y2_start = 5 - (s2 - 1) * 0.8
        for i in range(s1):
            for j in range(s2):
                y1 = y1_start + i * 1.6
                y2 = y2_start + j * 1.6
                ax.plot([l1 + 0.3, l2 - 0.3], [y1, y2], 'k-', alpha=0.3)
    
    # Dropout network 1
    ax = axes[1]
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 10)
    ax.axis('off')
    ax.set_title('With Dropout (p=0.5) - Sample 1', fontsize=14)
    
    # Draw network with dropout
    np.random.seed(42)
    for layer_idx, (x, size) in enumerate(zip(layers_pos, layer_sizes)):
        y_start = 5 - (size - 1) * 0.8
        for i in range(size):
            y = y_start + i * 1.6
            if layer_idx == 1 and np.random.random() < 0.5:  # Dropout in hidden layer
                ax.add_patch(Circle((x, y), 0.3, color='gray', ec='black', linestyle='--', alpha=0.3))
                ax.plot([x-0.3, x+0.3], [y-0.3, y+0.3], 'r-', linewidth=2)
                ax.plot([x-0.3, x+0.3], [y+0.3, y-0.3], 'r-', linewidth=2)
            else:
                color = ['lightgreen', 'lightblue', 'lightcoral'][layer_idx]
                ax.add_patch(Circle((x, y), 0.3, color=color, ec='black'))
    
    # Dropout network 2
    ax = axes[2]
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 10)
    ax.axis('off')
    ax.set_title('With Dropout (p=0.5) - Sample 2', fontsize=14)
    
    # Different dropout pattern
    np.random.seed(123)
    for layer_idx, (x, size) in enumerate(zip(layers_pos, layer_sizes)):
        y_start = 5 - (size - 1) * 0.8
        for i in range(size):
            y = y_start + i * 1.6
            if layer_idx == 1 and np.random.random() < 0.5:
                ax.add_patch(Circle((x, y), 0.3, color='gray', ec='black', linestyle='--', alpha=0.3))
                ax.plot([x-0.3, x+0.3], [y-0.3, y+0.3], 'r-', linewidth=2)
                ax.plot([x-0.3, x+0.3], [y+0.3, y-0.3], 'r-', linewidth=2)
            else:
                color = ['lightgreen', 'lightblue', 'lightcoral'][layer_idx]
                ax.add_patch(Circle((x, y), 0.3, color=color, ec='black'))
    
    plt.tight_layout()
    plt.show()

visualize_dropout()

Dropout benefits:
- Forces network to be robust (can't rely on specific neurons)
- Acts like training multiple networks
- Simple but effective regularization

<a name="early-stopping"></a>
### Early Stopping

Monitor validation loss and stop when it starts increasing:

In [ ]:
def visualize_early_stopping():
    epochs = np.arange(0, 200)
    
    # Simulate training and validation loss
    train_loss = 0.8 * np.exp(-0.03 * epochs) + 0.1 + 0.02 * np.random.randn(len(epochs)).cumsum() / len(epochs)
    val_loss = 0.8 * np.exp(-0.025 * epochs) + 0.15 + 0.001 * epochs + 0.03 * np.random.randn(len(epochs)).cumsum() / len(epochs)
    
    # Find early stopping point
    patience = 10
    best_epoch = np.argmin(val_loss[:100])
    stop_epoch = best_epoch + patience
    
    plt.figure(figsize=(10, 6))
    plt.plot(epochs, train_loss, 'b-', label='Training Loss', linewidth=2)
    plt.plot(epochs, val_loss, 'r-', label='Validation Loss', linewidth=2)
    plt.axvline(x=best_epoch, color='g', linestyle='--', label=f'Best Model (epoch {best_epoch})')
    plt.axvline(x=stop_epoch, color='orange', linestyle='--', label=f'Early Stop (epoch {stop_epoch})')
    
    # Highlight overfitting region
    plt.axvspan(stop_epoch, 200, alpha=0.2, color='red', label='Overfitting Region')
    
    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('Loss', fontsize=12)
    plt.title('Early Stopping to Prevent Overfitting', fontsize=14)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

visualize_early_stopping()

<a name="network-architecture-design"></a>
## Network Architecture Design

<a name="how-deep-should-we-go"></a>
### How Deep Should We Go?

Depth allows hierarchical feature learning:
- **Shallow networks (1-2 hidden layers)**: Simple patterns
- **Medium networks (3-5 layers)**: Most practical applications
- **Deep networks (10+ layers)**: Complex hierarchical features

<a name="how-wide-should-layers-be"></a>
### How Wide Should Layers Be?

Width affects capacity:
- **Too narrow**: Underfitting (information bottleneck)
- **Too wide**: Overfitting, computational cost
- **Rule of thumb**: Start with layers between input and output size

<a name="architecture-patterns"></a>
### Architecture Patterns

Common patterns:

In [ ]:
def visualize_architecture_patterns():
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    architectures = [
        ([4, 8, 4, 1], 'Expanding-Contracting'),
        ([4, 3, 2, 1], 'Funnel'),
        ([4, 4, 4, 1], 'Constant Width')
    ]
    
    for idx, (layers, name) in enumerate(architectures):
        ax = axes[idx]
        ax.set_xlim(0, 10)
        ax.set_ylim(0, 10)
        ax.axis('off')
        ax.set_title(name, fontsize=14)
        
        # Calculate layer positions
        n_layers = len(layers)
        x_positions = np.linspace(2, 8, n_layers)
        
        # Draw layers
        for i, (x, size) in enumerate(zip(x_positions, layers)):
            y_start = 5 - (size - 1) * 0.4
            
            for j in range(size):
                y = y_start + j * 0.8
                color = plt.cm.viridis(i / (n_layers - 1))
                ax.add_patch(Circle((x, y), 0.2, color=color, ec='black'))
            
            # Layer label
            ax.text(x, 1, f'{size}', ha='center', fontsize=12, weight='bold')
        
        # Draw connections
        for i in range(n_layers - 1):
            x1, x2 = x_positions[i], x_positions[i+1]
            s1, s2 = layers[i], layers[i+1]
            
            y1_start = 5 - (s1 - 1) * 0.4
            y2_start = 5 - (s2 - 1) * 0.4
            
            # Sample connections to avoid clutter
            for j in range(0, s1, max(1, s1//3)):
                for k in range(0, s2, max(1, s2//3)):
                    y1 = y1_start + j * 0.8
                    y2 = y2_start + k * 0.8
                    ax.plot([x1 + 0.2, x2 - 0.2], [y1, y2], 'k-', alpha=0.2)
    
    plt.suptitle('Common Architecture Patterns', fontsize=16)
    plt.tight_layout()
    plt.show()

visualize_architecture_patterns()

<a name="conclusion"></a>
## Conclusion

<a name="key-takeaways"></a>
### Key Takeaways

We've covered the fundamental concepts of neural networks:

1. **Architecture**: Networks of interconnected neurons create powerful function approximators
2. **Activation Functions**: Non-linearity enables learning complex patterns
3. **Forward Propagation**: Information flows layer by layer to make predictions
4. **Backpropagation**: Gradients flow backward to update weights
5. **Training**: Optimizers, scheduling, and regularization improve learning
6. **Design**: Architecture choices affect capacity and performance

Neural networks extend logistic regression by:
- Adding hidden layers for feature learning
- Enabling non-linear decision boundaries
- Learning hierarchical representations
- Scaling to complex problems

<a name="looking-ahead-to-lesson-3b"></a>
### Looking Ahead to Lesson 3B

In the next lesson, we'll implement these concepts in PyTorch:
- Building multi-layer networks with `nn.Module`
- Using PyTorch's automatic differentiation
- Implementing dropout and regularization
- Training on real medical data
- Comparing with our logistic regression baseline

We'll see how PyTorch makes implementing these complex ideas straightforward while maintaining the flexibility to experiment with architectures.

### Next lesson: [3B_neural_networks_pytorch.ipynb](./3b_neural_networks_pytorch.ipynb)

<a name="further-reading"></a>
### Further Reading

For deeper understanding:

1. **Foundational Texts**
   - [Deep Learning](https://www.deeplearningbook.org/) by Goodfellow, Bengio, and Courville
   - [Neural Networks and Deep Learning](http://neuralnetworksanddeeplearning.com/) by Michael Nielsen

2. **Practical Resources**
   - [PyTorch Tutorials](https://pytorch.org/tutorials/)
   - [Fast.ai Practical Deep Learning](https://course.fast.ai/)

3. **Advanced Topics**
   - [Batch Normalization](https://arxiv.org/abs/1502.03167)
   - [Dropout](https://jmlr.org/papers/v15/srivastava14a.html)
   - [Adam Optimizer](https://arxiv.org/abs/1412.6980)

Remember: Neural networks are powerful tools, but they're not always the best choice. Consider:
- Data size: Do you have enough data?
- Interpretability: Do you need to explain predictions?
- Computational resources: Can you afford training/inference?
- Problem complexity: Does it require deep learning?

The art is knowing when to use them!

### Thanks for learning!

This notebook is part of the Supervised Machine Learning from First Principles series.

© 2025 Powell-Clark Limited. Licensed under Apache License 2.0.

If you found this helpful, please cite as:
```
Powell-Clark (2025). Supervised Machine Learning from First Principles.
GitHub: https://github.com/powell-clark/supervised-machine-learning
```

Questions or feedback? Contact emmanuel@powellclark.com